In [ ]:
!pip install -q torch-geometric scikit-learn pandas numpy matplotlib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.9 MB/s eta 0:00:00


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

DATA_PATH = "/content/drive/MyDrive/TransXion/data/tx.csv"

print("Dataset exists:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    print("File size (MB):", round(os.path.getsize(DATA_PATH) / (1024**2), 2))

Dataset exists: True
File size (MB): 233.8


In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH, nrows=5)

print("Columns:")
print(df.columns.tolist())

print("\nSample transactions:")
display(df)

del df


Columns:
['Timestamp', 'From Bank', 'From Account', 'To Bank', 'To Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']

Sample transactions:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2025-01-01 00:00:29,3295718,A009852,2657408,A010012,4.40,CHF,8.06,AUD,Mobile,0
1,2025-01-01 00:00:40,3186407,A007397,3295718,A010077,58.45,CNY,58.51,CNY,Cash,0
2,2025-01-01 00:01:22,3186407,A007852,2793615,A003712,5576.27,CNY,5581.85,CNY,Mobile,0
3,2025-01-01 00:01:24,3862507,A005512,3071954,A010192,5.47,CNY,5.47,CNY,Mobile,0
4,2025-01-01 00:02:41,5472906,A001458,1527806,A010058,37.87,CNY,37.91,CNY,Card,0


In [ ]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/TransXion/data/tx.csv"

df = pd.read_csv(DATA_PATH, nrows=5)

print("Actual column names:")
for i, col in enumerate(df.columns):
    print(i, repr(col))

print("\nFirst five rows:")
display(df)


Actual column names:
0 'Timestamp'
1 'From Bank'
2 'From Account'
3 'To Bank'
4 'To Account'
5 'Amount Received'
6 'Receiving Currency'
7 'Amount Paid'
8 'Payment Currency'
9 'Payment Format'
10 'Is Laundering'

First five rows:


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2025-01-01 00:00:29,3295718,A009852,2657408,A010012,4.40,CHF,8.06,AUD,Mobile,0
1,2025-01-01 00:00:40,3186407,A007397,3295718,A010077,58.45,CNY,58.51,CNY,Cash,0
2,2025-01-01 00:01:22,3186407,A007852,2793615,A003712,5576.27,CNY,5581.85,CNY,Mobile,0
3,2025-01-01 00:01:24,3862507,A005512,3071954,A010192,5.47,CNY,5.47,CNY,Mobile,0
4,2025-01-01 00:02:41,5472906,A001458,1527806,A010058,37.87,CNY,37.91,CNY,Card,0


In [ ]:

import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/TransXion/data/tx.csv"

print("Loading transactions...")

df = pd.read_csv(DATA_PATH, low_memory=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["Is Laundering"] = df["Is Laundering"].astype("int8")

# Stable transaction ordering for equal timestamps
df["_order"] = np.arange(len(df), dtype=np.int32)
df = df.sort_values(
    ["Timestamp", "_order"], kind="mergesort"
).reset_index(drop=True)

# Use bank + account as the unique account identity
df["Sender"] = (
    df["From Bank"].astype(str) + "_" +
    df["From Account"].astype(str)
)
df["Receiver"] = (
    df["To Bank"].astype(str) + "_" +
    df["To Account"].astype(str)
)

# Map account IDs to integer graph-node IDs
accounts = pd.Index(
    pd.concat([df["Sender"], df["Receiver"]], ignore_index=True).unique()
)
account_to_id = {account: i for i, account in enumerate(accounts)}

df["SenderID"] = df["Sender"].map(account_to_id).astype("int32")
df["ReceiverID"] = df["Receiver"].map(account_to_id).astype("int32")

print("\nGraph preparation summary")
print("Transactions:", f"{len(df):,}")
print("Unique account nodes:", f"{len(accounts):,}")
print("Directed transaction edges:", f"{len(df):,}")
print("Laundering-labelled transactions:", f"{df['Is Laundering'].sum():,}")
print("Date range:", df["Timestamp"].min(), "to", df["Timestamp"].max())

print("\nFirst five mapped transactions:")
display(df[["Timestamp", "SenderID", "ReceiverID", "Is Laundering"]].head())


Loading transactions...

Graph preparation summary
Transactions: 3,029,170
Unique account nodes: 47,526
Directed transaction edges: 3,029,170
Laundering-labelled transactions: 4,641
Date range: 2025-01-01 00:00:29 to 2025-12-31 22:59:18

First five mapped transactions:


,Timestamp,SenderID,ReceiverID,Is Laundering
0,2025-01-01 00:00:29,0,24131,0
1,2025-01-01 00:00:40,1,47057,0
2,2025-01-01 00:01:22,2,3019,0
3,2025-01-01 00:01:24,3,39411,0
4,2025-01-01 00:02:41,4,33274,0


In [ ]:

# STEP 2: Create features and chronological splits

import numpy as np
import pandas as pd

# Basic transaction features
df["Hour"] = df["Timestamp"].dt.hour.astype("int8")
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek.astype("int8")

df["Amount Paid"] = pd.to_numeric(df["Amount Paid"], errors="coerce").fillna(0)
df["LogAmountPaid"] = np.log1p(df["Amount Paid"].clip(lower=0))

df["SameCurrency"] = (
    df["Payment Currency"].astype(str)
    == df["Receiving Currency"].astype(str)
).astype("int8")

# Chronological split: no random splitting
train_mask = df["Timestamp"] < "2025-09-01"
val_mask = (
    (df["Timestamp"] >= "2025-09-01")
    & (df["Timestamp"] < "2025-11-01")
)
test_mask = df["Timestamp"] >= "2025-11-01"

feature_cols = [
    "Hour", "DayOfWeek", "LogAmountPaid", "SameCurrency"
]

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df["Is Laundering"].to_numpy(dtype=np.int64)

train_idx = np.flatnonzero(train_mask.to_numpy())
val_idx = np.flatnonzero(val_mask.to_numpy())
test_idx = np.flatnonzero(test_mask.to_numpy())

print("Feature matrix:", X.shape)
print("Training transactions:", len(train_idx),
      "| positives:", int(y[train_idx].sum()))
print("Validation transactions:", len(val_idx),
      "| positives:", int(y[val_idx].sum()))
print("Test transactions:", len(test_idx),
      "| positives:", int(y[test_idx].sum()))
print("Feature columns:", feature_cols)


Feature matrix: (3029170, 4)
Training transactions: 2010061 | positives: 3237
Validation transactions: 504627 | positives: 671
Test transactions: 514482 | positives: 733
Feature columns: ['Hour', 'DayOfWeek', 'LogAmountPaid', 'SameCurrency']


In [ ]:

# STEP 3: Build a directed graph from training-period transactions only

import torch
from torch_geometric.data import Data

# Use training-period edges only
train_edges = df.iloc[train_idx]

edge_index = torch.tensor(
    np.vstack([
        train_edges["SenderID"].to_numpy(),
        train_edges["ReceiverID"].to_numpy()
    ]),
    dtype=torch.long
)

# Account features: number of historical transactions sent and received
num_nodes = len(accounts)

sent_count = np.bincount(
    train_edges["SenderID"].to_numpy(),
    minlength=num_nodes
)

received_count = np.bincount(
    train_edges["ReceiverID"].to_numpy(),
    minlength=num_nodes
)

node_features = np.column_stack([
    np.log1p(sent_count),
    np.log1p(received_count)
]).astype(np.float32)

# Normalize node features using training-graph statistics
node_mean = node_features.mean(axis=0)
node_std = node_features.std(axis=0)
node_std[node_std == 0] = 1.0
node_features = (node_features - node_mean) / node_std

graph = Data(
    x=torch.tensor(node_features, dtype=torch.float32),
    edge_index=edge_index,
    num_nodes=num_nodes
)

print("Training graph ready")
print("Nodes:", graph.num_nodes)
print("Directed edges:", graph.edge_index.shape[1])
print("Node feature shape:", tuple(graph.x.shape))
print("Edge index shape:", tuple(graph.edge_index.shape))
print("PyTorch Geometric:", __import__("torch_geometric").__version__)


Training graph ready
Nodes: 47526
Directed edges: 2010061
Node feature shape: (47526, 2)
Edge index shape: (2, 2010061)
PyTorch Geometric: 2.8.1


In [16]:

# STEP 4: Define two-layer GraphSAGE and transaction classifier

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class MoneyMuleGNN(nn.Module):
    def __init__(self, node_in=2, hidden=64, tx_in=8, dropout=0.3):
        super().__init__()

        self.sage1 = SAGEConv(node_in, hidden)
        self.sage2 = SAGEConv(hidden, hidden)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + tx_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )

        self.dropout = dropout

    def encode_nodes(self, x, edge_index):
        x = F.relu(self.sage1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.sage2(x, edge_index)
        return x

    def classify_transactions(self, embeddings, sender_ids, receiver_ids, tx_features):
        combined = torch.cat([
            embeddings[sender_ids],
            embeddings[receiver_ids],
            tx_features
        ], dim=1)
        return self.classifier(combined).squeeze(1)

model = MoneyMuleGNN().to(device)

print(model)
print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")


Device: cuda
MoneyMuleGNN(
  (sage1): SAGEConv(2, 64, aggr=mean)
  (sage2): SAGEConv(64, 64, aggr=mean)
  (classifier): Sequential(
    (0): Linear(in_features=136, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
  )
)
Model parameters: 17,409


In [17]:

required = [
    "df", "graph", "X_scaled", "sender_ids", "receiver_ids",
    "y_tensor", "train_idx", "val_idx", "test_idx"
]

for name in required:
    print(f"{name}: {'available' if name in globals() else 'MISSING'}")

print("Training transactions:", len(train_idx))
print("Validation transactions:", len(val_idx))
print("Test transactions:", len(test_idx))
print("Transaction feature count:", X_scaled.shape[1])


df: available
graph: available
X_scaled: available
sender_ids: available
receiver_ids: available
y_tensor: available
train_idx: available
val_idx: available
test_idx: available
Training transactions: 2010061
Validation transactions: 504627
Test transactions: 514482
Transaction feature count: 8


In [ ]:

# STEP 5: Prepare tensors and weighted loss

import torch
import numpy as np

# Transaction endpoint IDs
sender_ids = torch.tensor(
    df["SenderID"].to_numpy(), dtype=torch.long
)
receiver_ids = torch.tensor(
    df["ReceiverID"].to_numpy(), dtype=torch.long
)

# Transaction features and labels
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

# Split indices
train_idx_t = torch.tensor(train_idx, dtype=torch.long)
val_idx_t = torch.tensor(val_idx, dtype=torch.long)
test_idx_t = torch.tensor(test_idx, dtype=torch.long)

# Move graph and features to GPU
graph = graph.to(device)
X_tensor = X_tensor.to(device)
y_tensor = y_tensor.to(device)
sender_ids = sender_ids.to(device)
receiver_ids = receiver_ids.to(device)

train_idx_t = train_idx_t.to(device)
val_idx_t = val_idx_t.to(device)
test_idx_t = test_idx_t.to(device)

# Positive-class weighting, calculated from training labels only
train_labels = y_tensor[train_idx_t]
positive_count = train_labels.sum()
negative_count = len(train_labels) - positive_count

pos_weight = (negative_count / positive_count).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=1e-4
)

print("Training tensors ready")
print("Device:", device)
print("Positive training labels:", int(positive_count.item()))
print("Negative training labels:", int(negative_count.item()))
print("Positive class weight:", round(pos_weight.item(), 2))
print("Optimizer: Adam | Learning rate: 0.001")


Training tensors ready
Device: cuda
Positive training labels: 3237
Negative training labels: 2006824
Positive class weight: 619.96
Optimizer: Adam | Learning rate: 0.001


In [ ]:

# STEP 6: Memory-conscious GraphSAGE training

import gc
import torch
import numpy as np
from sklearn.metrics import average_precision_score

# Keep model and graph on GPU, as in the previous steps
model = model.to(device)
graph = graph.to(device)

# Training settings
epochs = 5
batch_size = 32768

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    # Compute account embeddings from the training-period graph
    embeddings = model.encode_nodes(graph.x, graph.edge_index)

    # Train the classifier in transaction mini-batches.
    # Accumulate gradients and update parameters once per epoch.
    total_loss = 0.0
    n_train = len(train_idx_t)

    for start in range(0, n_train, batch_size):
        batch_idx = train_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )

        # Scale each batch's loss so gradients represent the full dataset
        loss = criterion(logits, y_tensor[batch_idx])
        scaled_loss = loss * (len(batch_idx) / n_train)
        scaled_loss.backward()

        total_loss += loss.item() * len(batch_idx)

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Validation: do not update model parameters here
    model.eval()
    with torch.no_grad():
        val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_probs_parts = []

        for start in range(0, len(val_idx_t), batch_size):
            batch_idx = val_idx_t[start:start + batch_size]

            logits = model.classify_transactions(
                val_embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )
            val_probs_parts.append(torch.sigmoid(logits).cpu().numpy())

        val_probs = np.concatenate(val_probs_parts)
        val_labels = y[val_idx]

        val_ap = average_precision_score(val_labels, val_probs)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Weighted loss: {total_loss / n_train:.4f} | "
        f"Validation AP: {val_ap:.6f}"
    )

    del embeddings, val_embeddings, val_probs_parts
    gc.collect()
    torch.cuda.empty_cache()

print("Training run complete.")


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:

# GraphSAGE Model Training and Validation

import gc
import torch
import numpy as np
from sklearn.metrics import average_precision_score

epochs = 5
batch_size = 32768

# Training Loop
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    # Generate Account Embeddings
    embeddings = model.encode_nodes(graph.x, graph.edge_index)

    total_loss = 0.0
    n_train = len(train_idx_t)

    # Transaction-Level Classification
    for start in range(0, n_train, batch_size):
        batch_idx = train_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )

        loss = criterion(logits, y_tensor[batch_idx])
        scaled_loss = loss * (len(batch_idx) / n_train)
        scaled_loss.backward()

        total_loss += loss.item() * len(batch_idx)

    # Parameter Optimization
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    # Validation and Average Precision Evaluation
    model.eval()

    with torch.no_grad():
        val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_probs_parts = []

        for start in range(0, len(val_idx_t), batch_size):
            batch_idx = val_idx_t[start:start + batch_size]

            val_logits = model.classify_transactions(
                val_embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )

            val_probs_parts.append(
                torch.sigmoid(val_logits).cpu().numpy()
            )

        val_probs = np.concatenate(val_probs_parts)
        val_ap = average_precision_score(y[val_idx], val_probs)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Weighted Loss: {total_loss / n_train:.4f} | "
        f"Validation AP: {val_ap:.6f}"
    )

    del embeddings, val_embeddings, val_probs_parts
    gc.collect()
    torch.cuda.empty_cache()

print("Training complete.")


RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [ ]:

import gc
import torch
import numpy as np
from sklearn.metrics import average_precision_score

epochs = 5
batch_size = 32768

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    embeddings = model.encode_nodes(graph.x, graph.edge_index)
    train_logits_parts = []
    n_train = len(train_idx_t)

    # Training
    for start in range(0, n_train, batch_size):
        batch_idx = train_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )
        train_logits_parts.append(logits)

    train_logits = torch.cat(train_logits_parts)
    loss = criterion(train_logits, y_tensor[train_idx_t])
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_probs_parts = []

        for start in range(0, len(val_idx_t), batch_size):
            batch_idx = val_idx_t[start:start + batch_size]

            logits = model.classify_transactions(
                val_embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )
            val_probs_parts.append(torch.sigmoid(logits).cpu().numpy())

        val_probs = np.concatenate(val_probs_parts)
        val_ap = average_precision_score(y[val_idx], val_probs)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Weighted Loss: {loss.item():.4f} | "
        f"Validation AP: {val_ap:.6f}"
    )

    del embeddings, train_logits_parts, train_logits
    del val_embeddings, val_probs_parts
    gc.collect()
    torch.cuda.empty_cache()

print("Training complete.")


Epoch 1/5 | Weighted Loss: 1.5304 | Validation AP: 0.002051
Epoch 2/5 | Weighted Loss: 1.4574 | Validation AP: 0.003205
Epoch 3/5 | Weighted Loss: 1.3784 | Validation AP: 0.004599
Epoch 4/5 | Weighted Loss: 1.3322 | Validation AP: 0.006503
Epoch 5/5 | Weighted Loss: 1.2797 | Validation AP: 0.008470
Training complete.


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

model.eval()

with torch.no_grad():
    val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
    val_probs_parts = []

    for start in range(0, len(val_idx_t), batch_size):
        batch_idx = val_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            val_embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )
        val_probs_parts.append(torch.sigmoid(logits).cpu().numpy())

    val_probs = np.concatenate(val_probs_parts)

val_true = y[val_idx]
val_pred = (val_probs >= 0.5).astype(np.int8)

print("Validation Average Precision:", round(average_precision_score(val_true, val_probs), 6))
print("Precision:", round(precision_score(val_true, val_pred, zero_division=0), 4))
print("Recall:", round(recall_score(val_true, val_pred, zero_division=0), 4))
print("F1 Score:", round(f1_score(val_true, val_pred, zero_division=0), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(val_true, val_pred))
print("\nClassification Report:")
print(classification_report(val_true, val_pred, zero_division=0))


Validation Average Precision: 0.00847
Precision: 0.0034
Recall: 0.3249
F1 Score: 0.0067

Confusion Matrix:
[[439369  64587]
 [   453    218]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.87      0.93    503956
           1       0.00      0.32      0.01       671

    accuracy                           0.87    504627
   macro avg       0.50      0.60      0.47    504627
weighted avg       1.00      0.87      0.93    504627



In [ ]:

import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.01, 0.05, 0.10, 0.20, 0.30, 0.50, 0.70, 0.90]
results = []

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(np.int8)

    results.append({
        "Threshold": threshold,
        "Precision": precision_score(val_true, predictions, zero_division=0),
        "Recall": recall_score(val_true, predictions, zero_division=0),
        "F1": f1_score(val_true, predictions, zero_division=0),
        "Alerts": int(predictions.sum())
    })

threshold_results = pd.DataFrame(results)
display(threshold_results.round(4))


,Threshold,Precision,Recall,F1,Alerts
0,0.01,0.0013,1.0000,0.0027,504627
1,0.05,0.0013,1.0000,0.0027,504627
2,0.10,0.0013,1.0000,0.0027,504627
3,0.20,0.0013,1.0000,0.0027,504627
4,0.30,0.0013,0.9896,0.0026,501237
5,0.50,0.0034,0.3249,0.0067,64805
6,0.70,0.0000,0.0000,0.0000,7
7,0.90,0.0000,0.0000,0.0000,0


In [ ]:

score_summary = pd.DataFrame({
    "Class": ["Legitimate", "Laundering"],
    "Count": [
        int((val_true == 0).sum()),
        int((val_true == 1).sum())
    ],
    "Min Score": [
        val_probs[val_true == 0].min(),
        val_probs[val_true == 1].min()
    ],
    "Median Score": [
        np.median(val_probs[val_true == 0]),
        np.median(val_probs[val_true == 1])
    ],
    "Mean Score": [
        val_probs[val_true == 0].mean(),
        val_probs[val_true == 1].mean()
    ],
    "Max Score": [
        val_probs[val_true == 0].max(),
        val_probs[val_true == 1].max()
    ]
})

display(score_summary.round(6))
print("Unique predicted scores:", len(np.unique(val_probs)))
print("Overall score percentiles:")
print(np.percentile(val_probs, [0, 1, 10, 25, 50, 75, 90, 99, 100]))


,Class,Count,Min Score,Median Score,Mean Score,Max Score
0,Legitimate,503956,0.226371,0.426306,0.428090,0.728171
1,Laundering,671,0.250284,0.466759,0.468072,0.672194


Unique predicted scores: 473590
Overall score percentiles:
[0.22637108 0.30715779 0.35170631 0.38233617 0.4263536  0.47287081
 0.5078226  0.55635023 0.72817075]


In [ ]:

print("Model training diagnostics")
print("Current learning rate:", optimizer.param_groups[0]["lr"])
print("Graph nodes:", graph.num_nodes)
print("Training edges:", graph.edge_index.shape[1])
print("Training positives:", int(y[train_idx].sum()))
print("Validation positives:", int(y[val_idx].sum()))

for name, parameter in model.named_parameters():
    print(
        f"{name}: shape={tuple(parameter.shape)}, "
        f"gradient_available={parameter.grad is not None}"
    )


Model training diagnostics
Current learning rate: 0.001
Graph nodes: 47526
Training edges: 2010061
Training positives: 3237
Validation positives: 671
sage1.lin_l.weight: shape=(64, 2), gradient_available=True
sage1.lin_l.bias: shape=(64,), gradient_available=True
sage1.lin_r.weight: shape=(64, 2), gradient_available=True
sage2.lin_l.weight: shape=(64, 64), gradient_available=True
sage2.lin_l.bias: shape=(64,), gradient_available=True
sage2.lin_r.weight: shape=(64, 64), gradient_available=True
classifier.0.weight: shape=(64, 132), gradient_available=True
classifier.0.bias: shape=(64,), gradient_available=True
classifier.3.weight: shape=(1, 64), gradient_available=True
classifier.3.bias: shape=(1,), gradient_available=True


In [ ]:

from pathlib import Path

FEATURE_PATH = "/content/drive/MyDrive/TransXion/data/graph_features_v2.csv"

if Path(FEATURE_PATH).exists():
    hist = pd.read_csv(FEATURE_PATH, nrows=3)
    print("Historical features found.")
    print("Columns:", hist.columns.tolist())
    print("Continue with feature integration.")
else:
    print("Historical features file not found in Drive.")
    print("Continue with the existing GNN features.")


Historical features file not found in Drive.
Continue with the existing GNN features.


In [ ]:

from sklearn.preprocessing import StandardScaler
import torch
import numpy as np

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X[train_idx]).astype(np.float32)
X_val_scaled = scaler.transform(X[val_idx]).astype(np.float32)
X_test_scaled = scaler.transform(X[test_idx]).astype(np.float32)

X_scaled = np.empty_like(X, dtype=np.float32)
X_scaled[train_idx] = X_train_scaled
X_scaled[val_idx] = X_val_scaled
X_scaled[test_idx] = X_test_scaled

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)

model = MoneyMuleGNN().to(device)
optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=1e-4
)

train_labels = y_tensor[train_idx_t]
positive_count = train_labels.sum()
negative_count = len(train_labels) - positive_count
pos_weight = (negative_count / positive_count).to(device)

criterion = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Fresh model initialized.")
print("Scaled feature shape:", X_scaled.shape)
print("Positive class weight:", round(pos_weight.item(), 2))


Fresh model initialized.
Scaled feature shape: (3029170, 4)
Positive class weight: 619.96


In [ ]:

import gc
import torch
import numpy as np
from sklearn.metrics import average_precision_score

epochs = 5
batch_size = 32768

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    embeddings = model.encode_nodes(graph.x, graph.edge_index)
    train_logits_parts = []

    for start in range(0, len(train_idx_t), batch_size):
        batch_idx = train_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )
        train_logits_parts.append(logits)

    train_logits = torch.cat(train_logits_parts)
    loss = criterion(train_logits, y_tensor[train_idx_t])
    loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_probs_parts = []

        for start in range(0, len(val_idx_t), batch_size):
            batch_idx = val_idx_t[start:start + batch_size]

            logits = model.classify_transactions(
                val_embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )
            val_probs_parts.append(torch.sigmoid(logits).cpu().numpy())

        val_probs = np.concatenate(val_probs_parts)
        val_ap = average_precision_score(y[val_idx], val_probs)

    print(
        f"Epoch {epoch}/{epochs} | "
        f"Loss: {loss.item():.4f} | "
        f"Validation AP: {val_ap:.6f}"
    )

    del embeddings, train_logits_parts, train_logits
    del val_embeddings, val_probs_parts
    gc.collect()
    torch.cuda.empty_cache()

print("Training complete.")


Epoch 1/5 | Loss: 1.3521 | Validation AP: 0.005284
Epoch 2/5 | Loss: 1.3030 | Validation AP: 0.008231
Epoch 3/5 | Loss: 1.2487 | Validation AP: 0.011645
Epoch 4/5 | Loss: 1.2157 | Validation AP: 0.013217
Epoch 5/5 | Loss: 1.1752 | Validation AP: 0.013881
Training complete.


In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score
import pandas as pd
import numpy as np

thresholds = np.arange(0.10, 0.81, 0.05)
results = []

val_true = y[val_idx]

for threshold in thresholds:
    predictions = (val_probs >= threshold).astype(np.int8)

    results.append({
        "Threshold": round(float(threshold), 2),
        "Precision": precision_score(val_true, predictions, zero_division=0),
        "Recall": recall_score(val_true, predictions, zero_division=0),
        "F1": f1_score(val_true, predictions, zero_division=0),
        "Alerts": int(predictions.sum())
    })

threshold_results = pd.DataFrame(results)
display(threshold_results.round(4))


,Threshold,Precision,Recall,F1,Alerts
0,0.10,0.0013,1.0000,0.0027,504627
1,0.15,0.0013,1.0000,0.0027,504627
2,0.20,0.0013,1.0000,0.0027,504627
3,0.25,0.0013,1.0000,0.0027,504627
4,0.30,0.0013,1.0000,0.0027,504627
5,0.35,0.0013,0.9076,0.0026,459853
6,0.40,0.0016,0.8271,0.0032,343235
7,0.45,0.0020,0.7869,0.0040,264294
8,0.50,0.0021,0.7362,0.0043,231672
9,0.55,0.0023,0.5648,0.0046,163152


In [ ]:

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

threshold = 0.65
model.eval()

with torch.no_grad():
    test_embeddings = model.encode_nodes(graph.x, graph.edge_index)
    test_probs_parts = []

    for start in range(0, len(test_idx_t), batch_size):
        batch_idx = test_idx_t[start:start + batch_size]

        logits = model.classify_transactions(
            test_embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )
        test_probs_parts.append(torch.sigmoid(logits).cpu().numpy())

    test_probs = np.concatenate(test_probs_parts)

test_true = y[test_idx]
test_pred = (test_probs >= threshold).astype(np.int8)

print("Test Average Precision:", round(average_precision_score(test_true, test_probs), 6))
print("Threshold:", threshold)
print("Precision:", round(precision_score(test_true, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(test_true, test_pred, zero_division=0), 4))
print("F1 Score:", round(f1_score(test_true, test_pred, zero_division=0), 4))
print("Alerts:", int(test_pred.sum()))
print("\nConfusion Matrix:")
print(confusion_matrix(test_true, test_pred))
print("\nClassification Report:")
print(classification_report(test_true, test_pred, zero_division=0))


Test Average Precision: 0.00546
Threshold: 0.65
Precision: 0.0209
Recall: 0.0832
F1 Score: 0.0334
Alerts: 2915

Confusion Matrix:
[[510895   2854]
 [   672     61]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    513749
           1       0.02      0.08      0.03       733

    accuracy                           0.99    514482
   macro avg       0.51      0.54      0.52    514482
weighted avg       1.00      0.99      1.00    514482



In [1]:

print("Dataset loaded:", "df" in globals())
print("Graph available:", "graph" in globals())
print("GNN available:", "model" in globals())
print("Feature matrix available:", "X" in globals())
print("Training split available:", "train_idx" in globals())
print("Validation scores available:", "val_probs" in globals())
print("Test scores available:", "test_probs" in globals())


Dataset loaded: False
Graph available: False
GNN available: False
Feature matrix available: False
Training split available: False
Validation scores available: False
Test scores available: False


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

DATA_PATH = "/content/drive/MyDrive/TransXion/data/tx.csv"

print("Dataset exists:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    print("Dataset size (GB):", round(os.path.getsize(DATA_PATH) / 1e9, 3))

Dataset exists: True
Dataset size (GB): 0.245


In [4]:

import pandas as pd
import numpy as np

DATA_PATH = "/content/drive/MyDrive/TransXion/data/tx.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
df["Is Laundering"] = df["Is Laundering"].astype("int8")
df["_order"] = np.arange(len(df), dtype=np.int32)

df = df.sort_values(
    ["Timestamp", "_order"], kind="mergesort"
).reset_index(drop=True)

df["Sender"] = (
    df["From Bank"].astype(str) + "_" +
    df["From Account"].astype(str)
)
df["Receiver"] = (
    df["To Bank"].astype(str) + "_" +
    df["To Account"].astype(str)
)

accounts = pd.Index(
    pd.concat([df["Sender"], df["Receiver"]], ignore_index=True).unique()
)
account_to_id = {account: i for i, account in enumerate(accounts)}

df["SenderID"] = df["Sender"].map(account_to_id).astype("int32")
df["ReceiverID"] = df["Receiver"].map(account_to_id).astype("int32")

train_idx = np.flatnonzero((df["Timestamp"] < "2025-09-01").to_numpy())

val_idx = np.flatnonzero(
    ((df["Timestamp"] >= "2025-09-01") &
     (df["Timestamp"] < "2025-11-01")).to_numpy()
)

test_idx = np.flatnonzero((df["Timestamp"] >= "2025-11-01").to_numpy())

print("Transactions:", len(df))
print("Accounts:", len(accounts))
print("Training:", len(train_idx), "| Positives:", int(df.iloc[train_idx]["Is Laundering"].sum()))
print("Validation:", len(val_idx), "| Positives:", int(df.iloc[val_idx]["Is Laundering"].sum()))
print("Test:", len(test_idx), "| Positives:", int(df.iloc[test_idx]["Is Laundering"].sum()))


Transactions: 3029170
Accounts: 47526
Training: 2010061 | Positives: 3237
Validation: 504627 | Positives: 671
Test: 514482 | Positives: 733


In [5]:

df["Amount Paid"] = pd.to_numeric(
    df["Amount Paid"], errors="coerce"
).fillna(0).clip(lower=0)

df["Amount Received"] = pd.to_numeric(
    df["Amount Received"], errors="coerce"
).fillna(0).clip(lower=0)

df["Hour"] = df["Timestamp"].dt.hour
df["DayOfWeek"] = df["Timestamp"].dt.dayofweek
df["LogAmountPaid"] = np.log1p(df["Amount Paid"])

df["SameCurrency"] = (
    df["Payment Currency"].astype(str) ==
    df["Receiving Currency"].astype(str)
).astype("int8")

df["Past Sent Count"] = df.groupby("Sender", sort=False).cumcount()
df["Past Received Count"] = df.groupby("Receiver", sort=False).cumcount()

df["Past Sent Amount"] = (
    df.groupby(["Sender", "Payment Currency"], sort=False)["Amount Paid"].cumsum()
    - df["Amount Paid"]
)

df["Past Received Amount"] = (
    df.groupby(["Receiver", "Receiving Currency"], sort=False)["Amount Received"].cumsum()
    - df["Amount Received"]
)

feature_cols = [
    "Hour",
    "DayOfWeek",
    "LogAmountPaid",
    "SameCurrency",
    "Past Sent Count",
    "Past Received Count",
    "Past Sent Amount",
    "Past Received Amount",
]

print("Feature count:", len(feature_cols))
print("Feature matrix shape:", df[feature_cols].shape)
print("Missing feature values:", int(df[feature_cols].isna().sum().sum()))
print(df[feature_cols].head())


Feature count: 8
Feature matrix shape: (3029170, 8)
Missing feature values: 0
   Hour  DayOfWeek  LogAmountPaid  SameCurrency  Past Sent Count  \
0     0          2       2.203869             0                0   
1     0          2       4.086144             1                0   
2     0          2       8.627455             1                0   
3     0          2       1.867176             1                0   
4     0          2       3.661251             1                0   

   Past Received Count  Past Sent Amount  Past Received Amount  
0                    0               0.0                   0.0  
1                    0               0.0                   0.0  
2                    0               0.0                   0.0  
3                    0               0.0                   0.0  
4                    0               0.0                   0.0  


In [6]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = df.loc[train_idx, feature_cols].to_numpy(dtype=np.float32)
scaler.fit(X_train)

X_scaled = scaler.transform(
    df[feature_cols].to_numpy(dtype=np.float32)
).astype(np.float32)

print("Scaled feature matrix:", X_scaled.shape)
print("All values finite:", bool(np.isfinite(X_scaled).all()))
print("Training feature means:", np.round(X_scaled[train_idx].mean(axis=0), 3))


Scaled feature matrix: (3029170, 8)
All values finite: True
Training feature means: [ 0. -0.  0. -0.  0.  0.  0. -0.]


In [7]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_edges = df.iloc[train_idx]

edge_index = torch.tensor(
    np.vstack([
        train_edges["SenderID"].to_numpy(),
        train_edges["ReceiverID"].to_numpy()
    ]),
    dtype=torch.long
)

num_nodes = len(accounts)

sent_count = np.bincount(
    train_edges["SenderID"].to_numpy(), minlength=num_nodes
)
received_count = np.bincount(
    train_edges["ReceiverID"].to_numpy(), minlength=num_nodes
)

node_features = np.column_stack([
    np.log1p(sent_count),
    np.log1p(received_count)
]).astype(np.float32)

node_mean = node_features.mean(axis=0)
node_std = node_features.std(axis=0)
node_std[node_std == 0] = 1.0

node_features = (node_features - node_mean) / node_std

graph = Data(
    x=torch.tensor(node_features, dtype=torch.float32),
    edge_index=edge_index,
    num_nodes=num_nodes
).to(device)

class MoneyMuleGNN(nn.Module):
    def __init__(self, node_in=2, hidden=64, tx_in=8, dropout=0.3):
        super().__init__()
        self.sage1 = SAGEConv(node_in, hidden)
        self.sage2 = SAGEConv(hidden, hidden)
        self.sage3 = SAGEConv(hidden, hidden)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + tx_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
        self.dropout = dropout

    def encode_nodes(self, x, edge_index):
        x = F.relu(self.sage1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.sage2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.sage3(x, edge_index)
        return x

    def classify_transactions(self, embeddings, sender_ids, receiver_ids, tx_features):
        combined = torch.cat([
            embeddings[sender_ids],
            embeddings[receiver_ids],
            tx_features
        ], dim=1)
        return self.classifier(combined).squeeze(1)

model = MoneyMuleGNN().to(device)

print("Device:", device)
print("Graph nodes:", graph.num_nodes)
print("Training edges:", graph.edge_index.shape[1])
print("Node feature shape:", tuple(graph.x.shape))
print("Transaction feature count:", X_scaled.shape[1])
print("GNN layers:", 3)
print("Model parameters:", sum(p.numel() for p in model.parameters()))


ModuleNotFoundError: No module named 'torch_geometric'

In [8]:

%pip install torch-geometric


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.6 MB/s eta 0:00:00


In [9]:

import torch
import torch_geometric
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("CUDA available:", torch.cuda.is_available())



PyTorch: 2.11.0+cu130
PyTorch Geometric: 2.8.1
CUDA available: True


In [10]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_edges = df.iloc[train_idx]

edge_index = torch.tensor(
    np.vstack([
        train_edges["SenderID"].to_numpy(),
        train_edges["ReceiverID"].to_numpy()
    ]),
    dtype=torch.long
)

num_nodes = len(accounts)

sent_count = np.bincount(
    train_edges["SenderID"].to_numpy(), minlength=num_nodes
)
received_count = np.bincount(
    train_edges["ReceiverID"].to_numpy(), minlength=num_nodes
)

node_features = np.column_stack([
    np.log1p(sent_count),
    np.log1p(received_count)
]).astype(np.float32)

node_mean = node_features.mean(axis=0)
node_std = node_features.std(axis=0)
node_std[node_std == 0] = 1.0

node_features = (node_features - node_mean) / node_std

graph = Data(
    x=torch.tensor(node_features, dtype=torch.float32),
    edge_index=edge_index,
    num_nodes=num_nodes
).to(device)

class MoneyMuleGNN(nn.Module):
    def __init__(self, node_in=2, hidden=64, tx_in=8, dropout=0.3):
        super().__init__()
        self.sage1 = SAGEConv(node_in, hidden)
        self.sage2 = SAGEConv(hidden, hidden)
        self.sage3 = SAGEConv(hidden, hidden)

        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + tx_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
        self.dropout = dropout

    def encode_nodes(self, x, edge_index):
        x = F.relu(self.sage1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = F.relu(self.sage2(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)

        x = self.sage3(x, edge_index)
        return x

    def classify_transactions(self, embeddings, sender_ids, receiver_ids, tx_features):
        combined = torch.cat([
            embeddings[sender_ids],
            embeddings[receiver_ids],
            tx_features
        ], dim=1)
        return self.classifier(combined).squeeze(1)

model = MoneyMuleGNN().to(device)

print("Device:", device)
print("Graph nodes:", graph.num_nodes)
print("Training edges:", graph.edge_index.shape[1])
print("Node feature shape:", tuple(graph.x.shape))
print("Transaction feature count:", X_scaled.shape[1])
print("GNN layers:", 3)
print("Model parameters:", sum(p.numel() for p in model.parameters()))


Device: cuda
Graph nodes: 47526
Training edges: 2010061
Node feature shape: (47526, 2)
Transaction feature count: 8
GNN layers: 3
Model parameters: 25665


In [11]:

import random
import gc
import torch
import numpy as np

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

sender_ids = torch.tensor(
    df["SenderID"].to_numpy(), dtype=torch.long, device=device
)
receiver_ids = torch.tensor(
    df["ReceiverID"].to_numpy(), dtype=torch.long, device=device
)
X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor = torch.tensor(
    df["Is Laundering"].to_numpy(),
    dtype=torch.float32,
    device=device
)

train_pos = train_idx[df.iloc[train_idx]["Is Laundering"].to_numpy() == 1]
train_neg = train_idx[df.iloc[train_idx]["Is Laundering"].to_numpy() == 0]

rng = np.random.default_rng(SEED)
sampled_neg = rng.choice(
    train_neg, size=min(200_000, len(train_neg)), replace=False
)

sampled_train_idx = np.concatenate([train_pos, sampled_neg])
rng.shuffle(sampled_train_idx)

sampled_train_idx_t = torch.tensor(
    sampled_train_idx, dtype=torch.long, device=device
)

print("Training transactions sampled:", len(sampled_train_idx))
print("Positive examples retained:", len(train_pos))
print("Negative examples sampled:", len(sampled_neg))
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory allocated (GB):", round(torch.cuda.memory_allocated() / 1e9, 3))


Training transactions sampled: 203237
Positive examples retained: 3237
Negative examples sampled: 200000
GPU: Tesla T4
GPU memory allocated (GB): 0.194


In [12]:

from sklearn.metrics import average_precision_score
from torch_geometric.nn import SAGEConv
import torch.nn.functional as F

model = MoneyMuleGNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.001, weight_decay=1e-4
)

sampled_labels = y_tensor[sampled_train_idx_t]
positive_count = sampled_labels.sum()
negative_count = len(sampled_labels) - positive_count

pos_weight = negative_count / positive_count
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

val_idx_t = torch.tensor(val_idx, dtype=torch.long, device=device)

EPOCHS = 5
BATCH_SIZE = 8192

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    embeddings = model.encode_nodes(graph.x, graph.edge_index)
    epoch_loss = 0.0
    batches = 0

    for start in range(0, len(sampled_train_idx_t), BATCH_SIZE):
        batch_idx = sampled_train_idx_t[start:start + BATCH_SIZE]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )

        loss = criterion(logits, y_tensor[batch_idx])
        loss.backward(retain_graph=True)
        epoch_loss += loss.item()
        batches += 1

    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_logits_parts = []

        for start in range(0, len(val_idx_t), BATCH_SIZE):
            batch_idx = val_idx_t[start:start + BATCH_SIZE]
            logits = model.classify_transactions(
                val_embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )
            val_logits_parts.append(logits.cpu())

        val_probs = torch.sigmoid(torch.cat(val_logits_parts)).numpy()
        val_labels = df.iloc[val_idx]["Is Laundering"].to_numpy()
        val_ap = average_precision_score(val_labels, val_probs)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Loss: {epoch_loss / batches:.4f} | "
        f"Validation AP: {val_ap:.6f}"
    )

    del embeddings, val_embeddings
    torch.cuda.empty_cache()


Epoch 1/5 | Loss: 1.3399 | Validation AP: 0.003147
Epoch 2/5 | Loss: 1.3151 | Validation AP: 0.003538
Epoch 3/5 | Loss: 1.2876 | Validation AP: 0.003849
Epoch 4/5 | Loss: 1.2536 | Validation AP: 0.004207
Epoch 5/5 | Loss: 1.2245 | Validation AP: 0.004597


In [13]:

from sklearn.metrics import precision_score, recall_score, f1_score

val_labels = df.iloc[val_idx]["Is Laundering"].to_numpy()

print("Validation AP:", round(average_precision_score(val_labels, val_probs), 6))
print()

for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    predictions = (val_probs >= threshold).astype(int)

    print(
        f"Threshold {threshold:.1f} | "
        f"Alerts: {predictions.sum():,} | "
        f"Precision: {precision_score(val_labels, predictions, zero_division=0):.4f} | "
        f"Recall: {recall_score(val_labels, predictions, zero_division=0):.4f} | "
        f"F1: {f1_score(val_labels, predictions, zero_division=0):.4f}"
    )


Validation AP: 0.004597

Threshold 0.1 | Alerts: 504,627 | Precision: 0.0013 | Recall: 1.0000 | F1: 0.0027
Threshold 0.3 | Alerts: 504,626 | Precision: 0.0013 | Recall: 1.0000 | F1: 0.0027
Threshold 0.5 | Alerts: 185,693 | Precision: 0.0023 | Recall: 0.6304 | F1: 0.0045
Threshold 0.7 | Alerts: 0 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000
Threshold 0.9 | Alerts: 0 | Precision: 0.0000 | Recall: 0.0000 | F1: 0.0000


In [14]:

val_labels = df.iloc[val_idx]["Is Laundering"].to_numpy()

for label, name in [(0, "Legitimate"), (1, "Laundering")]:
    scores = val_probs[val_labels == label]

    print(
        f"{name}: "
        f"min={scores.min():.4f}, "
        f"median={np.median(scores):.4f}, "
        f"mean={scores.mean():.4f}, "
        f"max={scores.max():.4f}"
    )


Legitimate: min=0.2758, median=0.4727, mean=0.4724, max=0.6430
Laundering: min=0.3367, median=0.5177, mean=0.5051, max=0.6172


In [18]:
# STEP 5: Prepare training and validation data

import numpy as np
import torch

SEED = 42
rng = np.random.default_rng(SEED)

train_labels = df["Is Laundering"].to_numpy()

train_pos = train_idx[train_labels[train_idx] == 1]
train_neg = train_idx[train_labels[train_idx] == 0]

sampled_neg = rng.choice(
    train_neg,
    size=min(200_000, len(train_neg)),
    replace=False
)

sampled_train_idx = np.concatenate([train_pos, sampled_neg])
rng.shuffle(sampled_train_idx)

sampled_train_idx_t = torch.tensor(
    sampled_train_idx, dtype=torch.long, device=device
)

val_idx_t = torch.tensor(val_idx, dtype=torch.long, device=device)

pos_count = int(train_labels[sampled_train_idx].sum())
neg_count = len(sampled_train_idx) - pos_count

print("Sampled training transactions:", len(sampled_train_idx))
print("Laundering transactions:", pos_count)
print("Legitimate transactions:", neg_count)
print("Validation transactions:", len(val_idx))
print("Device:", device)

Sampled training transactions: 203237
Laundering transactions: 3237
Legitimate transactions: 200000
Validation transactions: 504627
Device: cuda


In [19]:
# STEP 6: Initialize model and training configuration

import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score

torch.manual_seed(42)
np.random.seed(42)

model = MoneyMuleGNN(
    node_in=2,
    hidden=64,
    tx_in=8,
    dropout=0.3
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

pos_weight = torch.tensor(
    neg_count / pos_count,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print("Model initialized")
print("Device:", device)
print("Positive class weight:", round(pos_weight.item(), 2))
print("Training batches will use the sampled training data.")

Model initialized
Device: cuda
Positive class weight: 61.79
Training batches will use the sampled training data.


In [20]:
# STEP 7: Train GraphSAGE transaction classifier

import copy
import numpy as np
import torch
from sklearn.metrics import average_precision_score

BATCH_SIZE = 8192
EPOCHS = 5

best_val_ap = -1.0
best_state = None
history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()

    embeddings = model.encode_nodes(graph.x, graph.edge_index)

    # Accumulate gradients across transaction batches
    for start in range(0, len(sampled_train_idx_t), BATCH_SIZE):
        batch_idx = sampled_train_idx_t[start:start + BATCH_SIZE]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )

        labels = y_tensor[batch_idx]
        loss = criterion(logits, labels)

        # Scale batch loss to obtain the mean over all sampled transactions
        scaled_loss = loss * (len(batch_idx) / len(sampled_train_idx_t))
        scaled_loss.backward(retain_graph=True)

    optimizer.step()

    # Validation
    model.eval()
    with torch.no_grad():
        embeddings = model.encode_nodes(graph.x, graph.edge_index)
        val_logits = []

        for start in range(0, len(val_idx_t), BATCH_SIZE):
            batch_idx = val_idx_t[start:start + BATCH_SIZE]

            logits = model.classify_transactions(
                embeddings,
                sender_ids[batch_idx],
                receiver_ids[batch_idx],
                X_tensor[batch_idx]
            )
            val_logits.append(logits.cpu())

        val_scores = torch.cat(val_logits).numpy()
        val_labels = y_tensor[val_idx_t].cpu().numpy()
        val_ap = average_precision_score(val_labels, val_scores)

    history.append({"epoch": epoch + 1, "validation_AP": val_ap})
    print(f"Epoch {epoch + 1}/{EPOCHS} | Validation AP: {val_ap:.6f}")

    if val_ap > best_val_ap:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model.state_dict())

model.load_state_dict(best_state)

print("\nBest validation AP:", round(best_val_ap, 6))
print("Best model restored.")

Epoch 1/5 | Validation AP: 0.004016
Epoch 2/5 | Validation AP: 0.004332
Epoch 3/5 | Validation AP: 0.004603
Epoch 4/5 | Validation AP: 0.004769
Epoch 5/5 | Validation AP: 0.004724

Best validation AP: 0.004769
Best model restored.


In [21]:
# STEP 8: Inspect validation score distributions

model.eval()

with torch.no_grad():
    embeddings = model.encode_nodes(graph.x, graph.edge_index)
    all_scores = []

    for start in range(0, len(val_idx_t), 8192):
        batch_idx = val_idx_t[start:start + 8192]

        logits = model.classify_transactions(
            embeddings,
            sender_ids[batch_idx],
            receiver_ids[batch_idx],
            X_tensor[batch_idx]
        )
        all_scores.append(torch.sigmoid(logits).cpu())

val_probs = torch.cat(all_scores).numpy()
val_y = y_tensor[val_idx_t].cpu().numpy().astype(int)

print("Legitimate transaction scores:")
print(np.percentile(val_probs[val_y == 0], [0, 25, 50, 75, 100]))

print("\nLaundering transaction scores:")
print(np.percentile(val_probs[val_y == 1], [0, 25, 50, 75, 100]))

print("\nValidation AP:", average_precision_score(val_y, val_probs))

Legitimate transaction scores:
[0.25890476 0.40548453 0.46188207 0.54119265 0.71427375]

Laundering transaction scores:
[0.32599789 0.48263542 0.54020798 0.57336727 0.66499841]

Validation AP: 0.004768546230733706


In [22]:
# STEP 9: Train a transaction-only baseline

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

X_train_sample = X_scaled[sampled_train_idx]
y_train_sample = train_labels[sampled_train_idx]

X_val = X_scaled[val_idx]
y_val = train_labels[val_idx]

baseline = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

baseline.fit(X_train_sample, y_train_sample)

baseline_val_scores = baseline.predict_proba(X_val)[:, 1]
baseline_ap = average_precision_score(y_val, baseline_val_scores)

print("Transaction-only validation AP:", round(baseline_ap, 6))
print("GNN validation AP:", round(best_val_ap, 6))

Transaction-only validation AP: 0.017559
GNN validation AP: 0.004769


In [23]:

# STEP 10: Compare models at fixed alert budgets

import numpy as np
import pandas as pd

def evaluate_alert_budget(y_true, scores, model_name):
    rows = []
    n = len(y_true)
    total_positive = int(np.sum(y_true))

    for rate in [0.001, 0.0025, 0.005, 0.01]:
        k = max(1, int(np.ceil(n * rate)))
        top_idx = np.argpartition(scores, -k)[-k:]
        tp = int(np.sum(y_true[top_idx]))

        rows.append({
            "Model": model_name,
            "Alert rate": f"{rate:.2%}",
            "Alerts": k,
            "True positives": tp,
            "Precision": tp / k,
            "Recall": tp / total_positive
        })

    return rows

results = []
results += evaluate_alert_budget(
    y_val, baseline_val_scores, "Transaction-only"
)
results += evaluate_alert_budget(
    val_y, val_probs, "GraphSAGE GNN"
)

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}"))


           Model Alert rate  Alerts  True positives  Precision  Recall
Transaction-only      0.10%     505              32     0.0634  0.0477
Transaction-only      0.25%    1262              66     0.0523  0.0984
Transaction-only      0.50%    2524             107     0.0424  0.1595
Transaction-only      1.00%    5047             153     0.0303  0.2280
   GraphSAGE GNN      0.10%     505              17     0.0337  0.0253
   GraphSAGE GNN      0.25%    1262              29     0.0230  0.0432
   GraphSAGE GNN      0.50%    2524              34     0.0135  0.0507
   GraphSAGE GNN      1.00%    5047              57     0.0113  0.0849


In [24]:

# STEP 11: Validation permutation importance

from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

sample_size = min(100_000, len(val_idx))
sample_rows = rng.choice(len(val_idx), size=sample_size, replace=False)

X_imp = X_val[sample_rows].copy()
y_imp = y_val[sample_rows]

baseline_scores = baseline.predict_proba(X_imp)[:, 1]
base_ap = average_precision_score(y_imp, baseline_scores)

importance_rows = []

for j, feature in enumerate(feature_cols):
    X_shuffled = X_imp.copy()
    X_shuffled[:, j] = rng.permutation(X_shuffled[:, j])

    shuffled_scores = baseline.predict_proba(X_shuffled)[:, 1]
    shuffled_ap = average_precision_score(y_imp, shuffled_scores)

    importance_rows.append({
        "Feature": feature,
        "Baseline AP": base_ap,
        "AP after shuffle": shuffled_ap,
        "AP decrease": base_ap - shuffled_ap
    })

importance = pd.DataFrame(importance_rows)
importance = importance.sort_values("AP decrease", ascending=False)

print("Validation sample size:", sample_size)
print("Sample baseline AP:", round(base_ap, 6))
print(importance.to_string(index=False, float_format=lambda x: f"{x:.6f}"))


Validation sample size: 100000
Sample baseline AP: 0.014799
             Feature  Baseline AP  AP after shuffle  AP decrease
 Past Received Count     0.014799          0.001668     0.013131
     Past Sent Count     0.014799          0.006003     0.008796
    Past Sent Amount     0.014799          0.008349     0.006450
        SameCurrency     0.014799          0.009788     0.005011
Past Received Amount     0.014799          0.011407     0.003392
       LogAmountPaid     0.014799          0.012357     0.002442
           DayOfWeek     0.014799          0.013881     0.000918
                Hour     0.014799          0.014336     0.000464


In [25]:

# STEP 12: Historical activity feature ablation

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

activity_features = [
    "Past Received Count",
    "Past Sent Count",
    "Past Sent Amount",
    "Past Received Amount"
]

transaction_features = [
    "Hour",
    "DayOfWeek",
    "LogAmountPaid",
    "SameCurrency"
]

feature_indices = {
    name: feature_cols.index(name)
    for name in feature_cols
}

def run_feature_group(features, model_name):
    cols = [feature_indices[name] for name in features]

    X_tr = X_scaled[sampled_train_idx][:, cols]
    y_tr = train_labels[sampled_train_idx]
    X_va = X_scaled[val_idx][:, cols]
    y_va = train_labels[val_idx]

    clf = HistGradientBoostingClassifier(
        max_iter=100,
        learning_rate=0.1,
        max_leaf_nodes=31,
        random_state=42
    )

    clf.fit(X_tr, y_tr)
    scores = clf.predict_proba(X_va)[:, 1]
    ap = average_precision_score(y_va, scores)

    print(f"{model_name} validation AP: {ap:.6f}")
    return clf, scores, ap

activity_model, activity_scores, activity_ap = run_feature_group(
    activity_features, "Activity-only"
)

transaction_model, transaction_scores, transaction_ap = run_feature_group(
    transaction_features, "Transaction-signals-only"
)

print(f"\nAll-eight-feature model AP: {baseline_ap:.6f}")


Activity-only validation AP: 0.017593
Transaction-signals-only validation AP: 0.002060

All-eight-feature model AP: 0.017559


In [27]:
# STEP 13: Compare activity-only and all-feature models

from sklearn.metrics import average_precision_score
import numpy as np
import pandas as pd

def alert_metrics(y_true, scores, budget):
    k = int(np.ceil(len(y_true) * budget))
    top = np.argpartition(scores, -k)[-k:]
    tp = int(y_true[top].sum())
    total_positive = int(y_true.sum())

    return {
        "Alert budget": f"{budget:.2%}",
        "Alerts": k,
        "True positives": tp,
        "Precision": tp / k,
        "Recall": tp / total_positive
    }

rows = []

for name, scores in [
    ("Activity-only", activity_scores),
    ("All eight features", baseline_val_scores)
]:
    for budget in [0.0025, 0.005, 0.01]:
        row = alert_metrics(y_val, scores, budget)
        row["Model"] = name
        rows.append(row)

print(pd.DataFrame(rows).to_string(
    index=False,
    float_format=lambda x: f"{x:.4f}"
))

Alert budget  Alerts  True positives  Precision  Recall              Model
       0.25%    1262              60     0.0475  0.0894      Activity-only
       0.50%    2524              87     0.0345  0.1297      Activity-only
       1.00%    5047             130     0.0258  0.1937      Activity-only
       0.25%    1262              66     0.0523  0.0984 All eight features
       0.50%    2524             107     0.0424  0.1595 All eight features
       1.00%    5047             153     0.0303  0.2280 All eight features


In [28]:
print("Graph node features:", tuple(graph.x.shape))
print("Graph edges:", tuple(graph.edge_index.shape))
print("Account count:", len(accounts))
print("Transaction features:", feature_cols)
print("Training positives:", int(train_labels[train_idx].sum()))
print("Validation positives:", int(train_labels[val_idx].sum()))
print("Best GNN validation AP:", round(best_val_ap, 6))
print("Baseline validation AP:", round(baseline_ap, 6))

Graph node features: (47526, 2)
Graph edges: (2, 2010061)
Account count: 47526
Transaction features: ['Hour', 'DayOfWeek', 'LogAmountPaid', 'SameCurrency', 'Past Sent Count', 'Past Received Count', 'Past Sent Amount', 'Past Received Amount']
Training positives: 3237
Validation positives: 671
Best GNN validation AP: 0.004769
Baseline validation AP: 0.017559


In [29]:
# STEP 14: Build a time-respecting graph with richer node features

import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data

# January-April: historical graph context
context_mask = df["Timestamp"] < "2025-05-01"
context_df = df.loc[context_mask].copy()

# May-August: supervised training period
supervised_train_idx = np.flatnonzero(
    (
        (df["Timestamp"] >= "2025-05-01") &
        (df["Timestamp"] < "2025-09-01")
    ).to_numpy()
)

# Account activity features computed from the historical context only
num_nodes = len(accounts)

sent_count = np.bincount(
    context_df["SenderID"].to_numpy(),
    minlength=num_nodes
)
received_count = np.bincount(
    context_df["ReceiverID"].to_numpy(),
    minlength=num_nodes
)

sent_unique = (
    context_df.groupby("SenderID")["ReceiverID"]
    .nunique()
    .reindex(range(num_nodes), fill_value=0)
    .to_numpy()
)

received_unique = (
    context_df.groupby("ReceiverID")["SenderID"]
    .nunique()
    .reindex(range(num_nodes), fill_value=0)
    .to_numpy()
)

node_features_new = np.column_stack([
    np.log1p(sent_count),
    np.log1p(received_count),
    np.log1p(sent_unique),
    np.log1p(received_unique)
]).astype(np.float32)

node_mean_new = node_features_new.mean(axis=0)
node_std_new = node_features_new.std(axis=0)
node_std_new[node_std_new == 0] = 1.0

node_features_new = (
    (node_features_new - node_mean_new) / node_std_new
)

# Directed edges from January-April only
edge_index_new = torch.tensor(
    np.vstack([
        context_df["SenderID"].to_numpy(),
        context_df["ReceiverID"].to_numpy()
    ]),
    dtype=torch.long
)

graph_new = Data(
    x=torch.tensor(node_features_new, dtype=torch.float32),
    edge_index=edge_index_new,
    num_nodes=num_nodes
).to(device)

print("Historical context ends:", "2025-04-30")
print("Historical transactions:", len(context_df))
print("Supervised training transactions:", len(supervised_train_idx))
print("Supervised training positives:",
      int(df.iloc[supervised_train_idx]["Is Laundering"].sum()))
print("Validation transactions:", len(val_idx))
print("Validation positives:", int(df.iloc[val_idx]["Is Laundering"].sum()))
print("New node feature shape:", tuple(graph_new.x.shape))
print("New graph edge shape:", tuple(graph_new.edge_index.shape))

Historical context ends: 2025-04-30
Historical transactions: 970662
Supervised training transactions: 1039399
Supervised training positives: 1695
Validation transactions: 504627
Validation positives: 671
New node feature shape: (47526, 4)
New graph edge shape: (2, 970662)


In [30]:
# STEP 15: Prepare features for the corrected experiment

from sklearn.preprocessing import StandardScaler
import numpy as np
import torch

scaler_new = StandardScaler()

X_train_new = df.iloc[supervised_train_idx][feature_cols].to_numpy(
    dtype=np.float32
)

scaler_new.fit(X_train_new)

X_scaled_new = scaler_new.transform(
    df[feature_cols].to_numpy(dtype=np.float32)
).astype(np.float32)

X_tensor_new = torch.tensor(
    X_scaled_new, dtype=torch.float32, device=device
)

y_tensor_new = torch.tensor(
    df["Is Laundering"].to_numpy(dtype=np.float32),
    dtype=torch.float32,
    device=device
)

sender_ids_new = torch.tensor(
    df["SenderID"].to_numpy(), dtype=torch.long, device=device
)

receiver_ids_new = torch.tensor(
    df["ReceiverID"].to_numpy(), dtype=torch.long, device=device
)

print("Transaction feature shape:", X_scaled_new.shape)
print("Training feature shape:", X_scaled_new[supervised_train_idx].shape)
print("Validation feature shape:", X_scaled_new[val_idx].shape)
print("Features:", feature_cols)
print("Missing values:", int(np.isnan(X_scaled_new).sum()))

Transaction feature shape: (3029170, 8)
Training feature shape: (1039399, 8)
Validation feature shape: (504627, 8)
Features: ['Hour', 'DayOfWeek', 'LogAmountPaid', 'SameCurrency', 'Past Sent Count', 'Past Received Count', 'Past Sent Amount', 'Past Received Amount']
Missing values: 0


In [31]:
# STEP 16: Corrected transaction-only baseline

import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score

rng = np.random.default_rng(42)

train_labels = df.iloc[supervised_train_idx]["Is Laundering"].to_numpy()

positive_idx_new = supervised_train_idx[train_labels == 1]
negative_idx_new = supervised_train_idx[train_labels == 0]

sampled_negatives_new = rng.choice(
    negative_idx_new,
    size=min(200_000, len(negative_idx_new)),
    replace=False
)

sampled_train_idx_new = np.concatenate([
    positive_idx_new,
    sampled_negatives_new
])

rng.shuffle(sampled_train_idx_new)

baseline_new = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

baseline_new.fit(
    X_scaled_new[sampled_train_idx_new],
    y_tensor_new[sampled_train_idx_new].cpu().numpy().astype(int)
)

val_y_new = df.iloc[val_idx]["Is Laundering"].to_numpy()

baseline_val_probs_new = baseline_new.predict_proba(
    X_scaled_new[val_idx]
)[:, 1]

baseline_val_ap_new = average_precision_score(
    val_y_new,
    baseline_val_probs_new
)

print("Training positives:", len(positive_idx_new))
print("Training negatives sampled:", len(sampled_negatives_new))
print("Total sampled training rows:", len(sampled_train_idx_new))
print("Validation average precision:", round(baseline_val_ap_new, 6))

for budget in [0.001, 0.0025, 0.005, 0.01]:
    k = max(1, int(len(val_y_new) * budget))
    top_idx = np.argpartition(baseline_val_probs_new, -k)[-k:]
    tp = int(val_y_new[top_idx].sum())

    print(
        f"Alert budget {budget:.2%}: "
        f"alerts={k}, true positives={tp}, "
        f"precision={tp/k:.4f}, "
        f"recall={tp/max(1, int(val_y_new.sum())):.4f}"
    )

Training positives: 1695
Training negatives sampled: 200000
Total sampled training rows: 201695
Validation average precision: 0.017985
Alert budget 0.10%: alerts=504, true positives=35, precision=0.0694, recall=0.0522
Alert budget 0.25%: alerts=1261, true positives=65, precision=0.0515, recall=0.0969
Alert budget 0.50%: alerts=2523, true positives=105, precision=0.0416, recall=0.1565
Alert budget 1.00%: alerts=5046, true positives=141, precision=0.0279, recall=0.2101


In [32]:
# STEP 17: Train corrected GraphSAGE model

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from sklearn.metrics import average_precision_score
import numpy as np

class MoneyMuleGNNNew(nn.Module):
    def __init__(self, node_in=4, hidden=64, tx_in=8, dropout=0.3):
        super().__init__()
        self.sage1 = SAGEConv(node_in, hidden)
        self.sage2 = SAGEConv(hidden, hidden)
        self.classifier = nn.Sequential(
            nn.Linear(hidden * 2 + tx_in, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, sender_ids, receiver_ids, tx_features):
        h = F.relu(self.sage1(x, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.sage2(h, edge_index)

        combined = torch.cat([
            h[sender_ids],
            h[receiver_ids],
            tx_features
        ], dim=1)

        return self.classifier(combined).squeeze(1)


model_new = MoneyMuleGNNNew().to(device)

train_labels_new = df.iloc[supervised_train_idx]["Is Laundering"].to_numpy()
positive_idx_gnn = supervised_train_idx[train_labels_new == 1]
negative_idx_gnn = supervised_train_idx[train_labels_new == 0]

rng_gnn = np.random.default_rng(42)
negative_sample_gnn = rng_gnn.choice(
    negative_idx_gnn,
    size=min(200_000, len(negative_idx_gnn)),
    replace=False
)

gnn_train_idx = np.concatenate([positive_idx_gnn, negative_sample_gnn])
rng_gnn.shuffle(gnn_train_idx)

train_labels_sampled = df.iloc[gnn_train_idx]["Is Laundering"].to_numpy()
pos_weight_new = torch.tensor(
    [(train_labels_sampled == 0).sum() / max(1, (train_labels_sampled == 1).sum())],
    dtype=torch.float32,
    device=device
)

criterion_new = nn.BCEWithLogitsLoss(pos_weight=pos_weight_new)
optimizer_new = torch.optim.Adam(
    model_new.parameters(), lr=0.001, weight_decay=1e-4
)

train_idx_tensor = torch.tensor(gnn_train_idx, dtype=torch.long, device=device)
val_idx_tensor = torch.tensor(val_idx, dtype=torch.long, device=device)

best_val_ap_new = -1
best_state_new = None
batch_size_new = 8192
epochs_new = 5

for epoch in range(1, epochs_new + 1):
    model_new.train()
    optimizer_new.zero_grad()

    embeddings_loss = 0
    for start in range(0, len(gnn_train_idx), batch_size_new):
        batch_idx = train_idx_tensor[start:start + batch_size_new]

        logits = model_new(
            graph_new.x,
            graph_new.edge_index,
            sender_ids_new[batch_idx],
            receiver_ids_new[batch_idx],
            X_tensor_new[batch_idx]
        )
        loss = criterion_new(logits, y_tensor_new[batch_idx])
        loss.backward()
        optimizer_new.step()
        optimizer_new.zero_grad()

    model_new.eval()
    with torch.no_grad():
        val_logits = model_new(
            graph_new.x,
            graph_new.edge_index,
            sender_ids_new[val_idx_tensor],
            receiver_ids_new[val_idx_tensor],
            X_tensor_new[val_idx_tensor]
        )
        val_probs_gnn_new = torch.sigmoid(val_logits).cpu().numpy()

    val_ap_gnn_new = average_precision_score(val_y_new, val_probs_gnn_new)

    if val_ap_gnn_new > best_val_ap_new:
        best_val_ap_new = val_ap_gnn_new
        best_state_new = {
            k: v.detach().cpu().clone()
            for k, v in model_new.state_dict().items()
        }

    print(f"Epoch {epoch}/{epochs_new} | Validation AP: {val_ap_gnn_new:.6f}")

model_new.load_state_dict(best_state_new)
print("Best corrected GNN validation AP:", round(best_val_ap_new, 6))

Epoch 1/5 | Validation AP: 0.102678
Epoch 2/5 | Validation AP: 0.133679
Epoch 3/5 | Validation AP: 0.144898
Epoch 4/5 | Validation AP: 0.141986
Epoch 5/5 | Validation AP: 0.138577
Best corrected GNN validation AP: 0.144898


In [33]:
# STEP 18: Compare baseline and best GNN on validation

from sklearn.metrics import average_precision_score
import numpy as np
import torch

model_new.eval()

with torch.no_grad():
    best_val_logits_new = model_new(
        graph_new.x,
        graph_new.edge_index,
        sender_ids_new[val_idx_tensor],
        receiver_ids_new[val_idx_tensor],
        X_tensor_new[val_idx_tensor]
    )
    best_val_probs_new = torch.sigmoid(
        best_val_logits_new
    ).cpu().numpy()

gnn_ap_verified = average_precision_score(
    val_y_new, best_val_probs_new
)

print("Baseline validation AP:", round(baseline_val_ap_new, 6))
print("Best GNN validation AP:", round(gnn_ap_verified, 6))
print()
print("Alert-budget comparison:")

for budget in [0.001, 0.0025, 0.005, 0.01]:
    k = max(1, int(len(val_y_new) * budget))

    base_top = np.argpartition(
        baseline_val_probs_new, -k
    )[-k:]

    gnn_top = np.argpartition(
        best_val_probs_new, -k
    )[-k:]

    base_tp = int(val_y_new[base_top].sum())
    gnn_tp = int(val_y_new[gnn_top].sum())
    total_positive = int(val_y_new.sum())

    print(
        f"\nBudget: {budget:.2%} | Alerts: {k}"
        f"\nBaseline: TP={base_tp}, "
        f"Precision={base_tp/k:.4f}, "
        f"Recall={base_tp/total_positive:.4f}"
        f"\nGNN:      TP={gnn_tp}, "
        f"Precision={gnn_tp/k:.4f}, "
        f"Recall={gnn_tp/total_positive:.4f}"
    )

Baseline validation AP: 0.017985
Best GNN validation AP: 0.144898

Alert-budget comparison:

Budget: 0.10% | Alerts: 504
Baseline: TP=35, Precision=0.0694, Recall=0.0522
GNN:      TP=166, Precision=0.3294, Recall=0.2474

Budget: 0.25% | Alerts: 1261
Baseline: TP=65, Precision=0.0515, Recall=0.0969
GNN:      TP=269, Precision=0.2133, Recall=0.4009

Budget: 0.50% | Alerts: 2523
Baseline: TP=105, Precision=0.0416, Recall=0.1565
GNN:      TP=291, Precision=0.1153, Recall=0.4337

Budget: 1.00% | Alerts: 5046
Baseline: TP=141, Precision=0.0279, Recall=0.2101
GNN:      TP=317, Precision=0.0628, Recall=0.4724


In [34]:
# STEP 19: Save the best GNN checkpoint and validation predictions

import os
import numpy as np
import torch
from google.colab import drive

drive.mount('/content/drive')

save_dir = "/content/drive/MyDrive/TransXion/experiments"
os.makedirs(save_dir, exist_ok=True)

torch.save(
    model_new.state_dict(),
    os.path.join(save_dir, "graphsage_best.pt")
)

np.save(
    os.path.join(save_dir, "gnn_validation_probabilities.npy"),
    best_val_probs_new
)

np.save(
    os.path.join(save_dir, "validation_labels.npy"),
    val_y_new
)

print("Saved model and validation predictions to:", save_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved model and validation predictions to: /content/drive/MyDrive/TransXion/experiments


In [35]:
# STEP 20: Graph ablation — self-loop-only graph

from torch_geometric.utils import remove_self_loops
from sklearn.metrics import average_precision_score
import torch
import numpy as np

# Keep only self-loops: no information passes between different accounts
num_nodes_new = graph_new.num_nodes

self_loop_edges = torch.arange(
    num_nodes_new, device=device, dtype=torch.long
).repeat(2, 1)

graph_self_new = graph_new.clone()
graph_self_new.edge_index = self_loop_edges

model_self_new = MoneyMuleGNNNew().to(device)

optimizer_self_new = torch.optim.Adam(
    model_self_new.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_self_ap = -1
best_self_state = None

for epoch in range(1, 4):
    model_self_new.train()

    # Use the same sampled training rows and class weighting
    for start in range(0, len(gnn_train_idx), batch_size_new):
        batch_idx = train_idx_tensor[start:start + batch_size_new]

        optimizer_self_new.zero_grad()

        logits = model_self_new(
            graph_self_new.x,
            graph_self_new.edge_index,
            sender_ids_new[batch_idx],
            receiver_ids_new[batch_idx],
            X_tensor_new[batch_idx]
        )

        loss = criterion_new(logits, y_tensor_new[batch_idx])
        loss.backward()
        optimizer_self_new.step()

    model_self_new.eval()

    with torch.no_grad():
        logits_val = model_self_new(
            graph_self_new.x,
            graph_self_new.edge_index,
            sender_ids_new[val_idx_tensor],
            receiver_ids_new[val_idx_tensor],
            X_tensor_new[val_idx_tensor]
        )

        probs_self = torch.sigmoid(logits_val).cpu().numpy()

    ap_self = average_precision_score(val_y_new, probs_self)

    if ap_self > best_self_ap:
        best_self_ap = ap_self
        best_self_state = {
            k: v.detach().cpu().clone()
            for k, v in model_self_new.state_dict().items()
        }

    print(f"Epoch {epoch}/3 | Self-loop-only validation AP: {ap_self:.6f}")

print("\nFull-graph GNN validation AP:", round(best_val_ap_new, 6))
print("Self-loop-only GNN best validation AP:", round(best_self_ap, 6))

Epoch 1/3 | Self-loop-only validation AP: 0.093111
Epoch 2/3 | Self-loop-only validation AP: 0.102352
Epoch 3/3 | Self-loop-only validation AP: 0.110501

Full-graph GNN validation AP: 0.144898
Self-loop-only GNN best validation AP: 0.110501


In [36]:
# STEP 22: Create a shuffled-edge graph control

import torch
import numpy as np

torch.manual_seed(42)

edge_index_shuffled = graph_new.edge_index.clone()

# Preserve source nodes; randomly shuffle destination nodes
perm = torch.randperm(
    edge_index_shuffled.size(1),
    device=edge_index_shuffled.device
)

edge_index_shuffled[1] = edge_index_shuffled[1, perm]

graph_shuffled = graph_new.clone()
graph_shuffled.edge_index = edge_index_shuffled

print("Original graph nodes:", graph_new.num_nodes)
print("Original graph edges:", graph_new.edge_index.size(1))
print("Shuffled graph nodes:", graph_shuffled.num_nodes)
print("Shuffled graph edges:", graph_shuffled.edge_index.size(1))
print(
    "Edge index identical:",
    torch.equal(graph_new.edge_index, graph_shuffled.edge_index)
)

Original graph nodes: 47526
Original graph edges: 970662
Shuffled graph nodes: 47526
Shuffled graph edges: 970662
Edge index identical: False


In [37]:
# STEP 23: Train GNN using shuffled graph edges

from sklearn.metrics import average_precision_score
import torch
import numpy as np

model_shuffled = MoneyMuleGNNNew().to(device)

optimizer_shuffled = torch.optim.Adam(
    model_shuffled.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

best_shuffled_ap = -1
best_shuffled_state = None

for epoch in range(1, 4):
    model_shuffled.train()

    for start in range(0, len(gnn_train_idx), batch_size_new):
        batch_idx = train_idx_tensor[start:start + batch_size_new]

        optimizer_shuffled.zero_grad()

        logits = model_shuffled(
            graph_shuffled.x,
            graph_shuffled.edge_index,
            sender_ids_new[batch_idx],
            receiver_ids_new[batch_idx],
            X_tensor_new[batch_idx]
        )

        loss = criterion_new(logits, y_tensor_new[batch_idx])
        loss.backward()
        optimizer_shuffled.step()

    model_shuffled.eval()

    with torch.no_grad():
        logits_val = model_shuffled(
            graph_shuffled.x,
            graph_shuffled.edge_index,
            sender_ids_new[val_idx_tensor],
            receiver_ids_new[val_idx_tensor],
            X_tensor_new[val_idx_tensor]
        )

        probs_shuffled = torch.sigmoid(logits_val).cpu().numpy()

    ap_shuffled = average_precision_score(val_y_new, probs_shuffled)

    if ap_shuffled > best_shuffled_ap:
        best_shuffled_ap = ap_shuffled
        best_shuffled_state = {
            k: v.detach().cpu().clone()
            for k, v in model_shuffled.state_dict().items()
        }

    print(
        f"Epoch {epoch}/3 | "
        f"Shuffled-graph validation AP: {ap_shuffled:.6f}"
    )

print("\nFull-graph GNN AP:", round(best_val_ap_new, 6))
print("Self-loop-only GNN AP:", round(best_self_ap, 6))
print("Shuffled-graph GNN best AP:", round(best_shuffled_ap, 6))

Epoch 1/3 | Shuffled-graph validation AP: 0.075572
Epoch 2/3 | Shuffled-graph validation AP: 0.104231
Epoch 3/3 | Shuffled-graph validation AP: 0.122223

Full-graph GNN AP: 0.144898
Self-loop-only GNN AP: 0.110501
Shuffled-graph GNN best AP: 0.122223


In [38]:
# STEP 25: Save validation experiment summary

import os
import json

results = {
    "dataset": "TransXion",
    "historical_graph_period": "January-April 2025",
    "supervised_training_period": "May-August 2025",
    "validation_period": "September-October 2025",
    "baseline_validation_ap": float(baseline_val_ap_new),
    "full_graph_gnn_validation_ap": float(best_val_ap_new),
    "self_loop_gnn_validation_ap": float(best_self_ap),
    "shuffled_graph_gnn_validation_ap": float(best_shuffled_ap),
    "full_graph_gnn_alert_budget_results": {
        "0.10%": {"true_positives": 166, "precision": 0.3294, "recall": 0.2474},
        "0.25%": {"true_positives": 269, "precision": 0.2133, "recall": 0.4009},
        "0.50%": {"true_positives": 291, "precision": 0.1153, "recall": 0.4337},
        "1.00%": {"true_positives": 317, "precision": 0.0628, "recall": 0.4724}
    },
    "test_set_used_for_tuning": False
}

save_dir = "/content/drive/MyDrive/TransXion/experiments"
os.makedirs(save_dir, exist_ok=True)

result_path = os.path.join(save_dir, "validation_results.json")

with open(result_path, "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to:", result_path)
print(json.dumps(results, indent=2))

Results saved to: /content/drive/MyDrive/TransXion/experiments/validation_results.json
{
  "dataset": "TransXion",
  "historical_graph_period": "January-April 2025",
  "supervised_training_period": "May-August 2025",
  "validation_period": "September-October 2025",
  "baseline_validation_ap": 0.01798485398953597,
  "full_graph_gnn_validation_ap": 0.1448984290674087,
  "self_loop_gnn_validation_ap": 0.11050057506130159,
  "shuffled_graph_gnn_validation_ap": 0.12222316062643941,
  "full_graph_gnn_alert_budget_results": {
    "0.10%": {
      "true_positives": 166,
      "precision": 0.3294,
      "recall": 0.2474
    },
    "0.25%": {
      "true_positives": 269,
      "precision": 0.2133,
      "recall": 0.4009
    },
    "0.50%": {
      "true_positives": 291,
      "precision": 0.1153,
      "recall": 0.4337
    },
    "1.00%": {
      "true_positives": 317,
      "precision": 0.0628,
      "recall": 0.4724
    }
  },
  "test_set_used_for_tuning": false
}


In [39]:
# STEP 26: Final evaluation on November-December test data

import numpy as np
import torch
from sklearn.metrics import average_precision_score
from sklearn.ensemble import HistGradientBoostingClassifier

# Test period
test_y = df.iloc[test_idx]["Is Laundering"].to_numpy()

test_idx_tensor = torch.tensor(
    test_idx, dtype=torch.long, device=device
)

# Restore the best saved GraphSAGE checkpoint
model_new.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/TransXion/experiments/graphsage_best.pt",
        map_location="cpu",
        weights_only=True
    )
)
model_new.to(device)
model_new.eval()

with torch.no_grad():
    test_logits_gnn = model_new(
        graph_new.x,
        graph_new.edge_index,
        sender_ids_new[test_idx_tensor],
        receiver_ids_new[test_idx_tensor],
        X_tensor_new[test_idx_tensor]
    )

test_probs_gnn = torch.sigmoid(test_logits_gnn).cpu().numpy()

# Fit baseline on the same sampled May-August training rows
baseline_test = HistGradientBoostingClassifier(
    max_iter=100,
    learning_rate=0.1,
    max_leaf_nodes=31,
    random_state=42
)

baseline_test.fit(
    X_scaled_new[sampled_train_idx_new],
    df.iloc[sampled_train_idx_new]["Is Laundering"].to_numpy()
)

test_probs_baseline = baseline_test.predict_proba(
    X_scaled_new[test_idx]
)[:, 1]

# Overall ranking performance
gnn_test_ap = average_precision_score(test_y, test_probs_gnn)
baseline_test_ap = average_precision_score(test_y, test_probs_baseline)

print("Test transactions:", len(test_y))
print("Test positive transactions:", int(test_y.sum()))
print("GNN test AP:", round(gnn_test_ap, 6))
print("Baseline test AP:", round(baseline_test_ap, 6))

# Operational alert-budget comparison
print("\nTest alert-budget comparison:")

for budget in [0.001, 0.0025, 0.005, 0.01]:
    k = max(1, int(len(test_y) * budget))

    gnn_top = np.argpartition(test_probs_gnn, -k)[-k:]
    base_top = np.argpartition(test_probs_baseline, -k)[-k:]

    gnn_tp = int(test_y[gnn_top].sum())
    base_tp = int(test_y[base_top].sum())
    total_pos = max(1, int(test_y.sum()))

    print(
        f"\nBudget: {budget:.2%} | Alerts: {k}"
        f"\nGNN: TP={gnn_tp}, Precision={gnn_tp/k:.4f}, "
        f"Recall={gnn_tp/total_pos:.4f}"
        f"\nBaseline: TP={base_tp}, Precision={base_tp/k:.4f}, "
        f"Recall={base_tp/total_pos:.4f}"
    )

Test transactions: 514482
Test positive transactions: 733
GNN test AP: 0.143055
Baseline test AP: 0.009234

Test alert-budget comparison:

Budget: 0.10% | Alerts: 514
GNN: TP=175, Precision=0.3405, Recall=0.2387
Baseline: TP=14, Precision=0.0272, Recall=0.0191

Budget: 0.25% | Alerts: 1286
GNN: TP=288, Precision=0.2240, Recall=0.3929
Baseline: TP=32, Precision=0.0249, Recall=0.0437

Budget: 0.50% | Alerts: 2572
GNN: TP=302, Precision=0.1174, Recall=0.4120
Baseline: TP=80, Precision=0.0311, Recall=0.1091

Budget: 1.00% | Alerts: 5144
GNN: TP=320, Precision=0.0622, Recall=0.4366
Baseline: TP=109, Precision=0.0212, Recall=0.1487


In [40]:
# STEP 27: Save final test results

import json
from pathlib import Path

results_dir = Path("/content/drive/MyDrive/TransXion/experiments")
results_dir.mkdir(parents=True, exist_ok=True)

final_results = {
    "dataset": "TransXion",
    "historical_graph_period": "January-April 2025",
    "supervised_training_period": "May-August 2025",
    "validation_period": "September-October 2025",
    "test_period": "November-December 2025",
    "test_transactions": int(len(test_y)),
    "test_laundering_transactions": int(test_y.sum()),
    "gnn_test_average_precision": float(gnn_test_ap),
    "baseline_test_average_precision": float(baseline_test_ap),
    "gnn_to_baseline_ap_ratio": float(gnn_test_ap / baseline_test_ap),
    "test_used_for_model_tuning": False,
    "alert_budget_results": {
        "0.10%": {
            "alerts": 514,
            "gnn_true_positives": 175,
            "gnn_precision": 0.3405,
            "gnn_recall": 0.2387,
            "baseline_true_positives": 14,
            "baseline_precision": 0.0272,
            "baseline_recall": 0.0191
        },
        "0.25%": {
            "alerts": 1286,
            "gnn_true_positives": 288,
            "gnn_precision": 0.2240,
            "gnn_recall": 0.3929,
            "baseline_true_positives": 32,
            "baseline_precision": 0.0249,
            "baseline_recall": 0.0437
        },
        "0.50%": {
            "alerts": 2572,
            "gnn_true_positives": 302,
            "gnn_precision": 0.1174,
            "gnn_recall": 0.4120,
            "baseline_true_positives": 80,
            "baseline_precision": 0.0311,
            "baseline_recall": 0.1091
        },
        "1.00%": {
            "alerts": 5144,
            "gnn_true_positives": 320,
            "gnn_precision": 0.0622,
            "gnn_recall": 0.4366,
            "baseline_true_positives": 109,
            "baseline_precision": 0.0212,
            "baseline_recall": 0.1487
        }
    }
}

output_path = results_dir / "final_test_results.json"

with open(output_path, "w") as f:
    json.dump(final_results, f, indent=2)

print("Final results saved to:", output_path)
print("GNN test AP:", round(gnn_test_ap, 6))
print("Baseline test AP:", round(baseline_test_ap, 6))
print("GNN improvement:", round(gnn_test_ap / baseline_test_ap, 2), "x")

Final results saved to: /content/drive/MyDrive/TransXion/experiments/final_test_results.json
GNN test AP: 0.143055
Baseline test AP: 0.009234
GNN improvement: 15.49 x
